Bibliotecas

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, f1_score, accuracy_score)
import joblib
import json

Diretorios

In [11]:
# Diretórios
DATA_DIR = Path("/content")
MODELS_DIR = Path("/content")

# Criar diretório de modelos se não existir
MODELS_DIR.mkdir(exist_ok=True)

Dados

In [12]:
# Carregar dados consolidados
df = pd.read_csv(DATA_DIR / "df_clean.csv")

print(f"✓ Dataset carregado: {len(df)} linhas × {len(df.columns)} colunas")
print(f"  Anos: {sorted(df['ANO'].unique())}")
print(f"  Indicadores: IAN, IDA, IEG, IAA, IPS, IPP, IPV, INDE")
print()

# Verificar distribuição
print("Distribuição por ano:")
print(df['ANO'].value_counts().sort_index())
print()

✓ Dataset carregado: 3030 linhas × 20 colunas
  Anos: [np.int64(2022), np.int64(2023), np.int64(2024)]
  Indicadores: IAN, IDA, IEG, IAA, IPS, IPP, IPV, INDE

Distribuição por ano:
ANO
2022     860
2023    1014
2024    1156
Name: count, dtype: int64



Feature Enginnering

In [14]:
# Criar dataset de modelagem: usar 2022-2023 para prever 2024
# Dados 2022-2023 como features
df_train_years = df[df['ANO'].isin([2022, 2023])].copy()

# Dados 2024 como target
df_target_year = df[df['ANO'] == 2024][['RA', 'IAN']].copy()
df_target_year.rename(columns={'IAN': 'IAN_2024'}, inplace=True)

# Agregar features por aluno (média dos anos 2022-2023)
features_agg = df_train_years.groupby('RA').agg({
    'IAN': 'mean',
    'IDA': 'mean',
    'IEG': 'mean',
    'IAA': 'mean',
    'IPS': 'mean',
    'IPP': 'mean',
    'IPV': 'mean',
    'INDE': 'mean',
    'INDICADORES_MEDIA': 'mean',
    'IEG_IDA_RATIO': 'mean',
    'INDE_IDA_RATIO': 'mean',
    'DESALINHAMENTO_IAA_IDA': 'mean',
}).reset_index()

# Merge com target (2024)
df_model = pd.merge(features_agg, df_target_year, on='RA', how='inner')

print(f"Dataset de modelagem: {len(df_model)} alunos")
print(f"  Features: {len(df_model.columns) - 2} indicadores")
print()

# Target: 1 se IAN_2024 < 10 (em risco), 0 caso contrário
df_model['TARGET_RISCO'] = (df_model['IAN_2024'] < 10).astype(int)

print(f"Distribuição do target:")
print(f"  Não em risco (IAN ≥ 10): {(df_model['TARGET_RISCO'] == 0).sum()} ({(df_model['TARGET_RISCO'] == 0).mean():.1%})")
print(f"  Em risco (IAN < 10): {(df_model['TARGET_RISCO'] == 1).sum()} ({(df_model['TARGET_RISCO'] == 1).mean():.1%})")
print()

# Features para modelagem
features_list = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'INDE',
                 'INDICADORES_MEDIA', 'IEG_IDA_RATIO', 'INDE_IDA_RATIO',
                 'DESALINHAMENTO_IAA_IDA']

print(f"Features utilizadas ({len(features_list)}):")
for i, feat in enumerate(features_list, 1):
    print(f"  {i}. {feat}")
print()

Preparando dados para modelagem...
Dataset de modelagem: 769 alunos
  Features: 12 indicadores

Distribuição do target:
  Não em risco (IAN ≥ 10): 460 (59.8%)
  Em risco (IAN < 10): 309 (40.2%)

Features utilizadas (12):
  1. IAN
  2. IDA
  3. IEG
  4. IAA
  5. IPS
  6. IPP
  7. IPV
  8. INDE
  9. INDICADORES_MEDIA
  10. IEG_IDA_RATIO
  11. INDE_IDA_RATIO
  12. DESALINHAMENTO_IAA_IDA



 base de modelagem para prever o risco de defasagem dos alunos em 2024, usando o desempenho médio deles em 2022 e 2023 como variáveis explicativas.


Usa dados de 2022-2023 para criar features agregadas por aluno (média dos indicadores).
Junta com o IAN de 2024, que vira o alvo da previsão.
Define o target de risco: aluno é considerado em risco se IAN_2024 < 10.
Seleciona 12 indicadores educacionais para modelagem.

Resultado:
Base final com 769 alunos.
40,2% estão em risco de defasagem em 2024 e 59,8% não estão.
O modelo usará indicadores de desempenho, engajamento e alinhamento pedagógico para prever o risco.

O pipeline prepara dados históricos para construir um modelo preditivo de risco educacional futuro.
___________________________

Preparação dos dados

In [15]:
# Selecionar features e target
X = df_model[features_list].copy()
y = df_model['TARGET_RISCO'].copy()

# Preencher valores faltantes com a média
X = X.fillna(X.mean())

print(f"✓ X shape: {X.shape}")
print(f"✓ y shape: {y.shape}")
print(f"✓ Valores faltantes em X: {X.isnull().sum().sum()}")
print()

ETAPA 3: PREPARAÇÃO DOS DADOS

✓ X shape: (769, 12)
✓ y shape: (769,)
✓ Valores faltantes em X: 0



Treino X Treste

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {len(X_train)} alunos ({len(X_train)/len(X):.1%})")
print(f"Teste: {len(X_test)} alunos ({len(X_test)/len(X):.1%})")
print(f"Proporção em risco (treino): {y_train.mean():.1%}")
print(f"Proporção em risco (teste): {y_test.mean():.1%}")
print()

# Normalizar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Treino: 615 alunos (80.0%)
Teste: 154 alunos (20.0%)
Proporção em risco (treino): 40.2%
Proporção em risco (teste): 40.3%



O código divide a base em 80% para treino (615 alunos) e 20% para teste (154 alunos), mantendo a mesma proporção de alunos em risco (~40%) em ambos os conjuntos.
Depois, realiza a normalização das variáveis, preparando os dados para a modelagem.

Os dados foram separados de forma equilibrada e padronizados para treinar e avaliar o modelo de risco.

_____________________________________

Modelagem Preditiva -  Random Forest

In [22]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Predições
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_rf = accuracy_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"  Acurácia: {acc_rf:.1%}")
print(f"  ROC AUC: {roc_auc_rf:.4f}")
print()

# Cross-validation
cv_scores_rf = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"  Cross-Validation (5-fold) ROC AUC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")
print()

  Acurácia: 78.6%
  ROC AUC: 0.8436

  Cross-Validation (5-fold) ROC AUC: 0.8467 ± 0.0345



Acurácia: 78,6% - bom nível de acerto nas classificações.
ROC AUC: 0,84 - boa capacidade de diferenciar alunos em risco vs. não risco.
Cross-validation ROC AUC: 0,85 ± 0,03 - desempenho estável e consistente, sem sinais fortes de overfitting.
____________________________________

Modelagem Preditiva - Gradient Boost

In [23]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

# Predições
y_pred_gb = gb_model.predict(X_test_scaled)
y_pred_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_gb = accuracy_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_pred_proba_gb)

print(f" Gradient Boosting treinado")
print(f"  Acurácia: {acc_gb:.1%}")
print(f"  ROC AUC: {roc_auc_gb:.4f}")
print()

# Cross-validation
cv_scores_gb = cross_val_score(gb_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"  Cross-Validation (5-fold) ROC AUC: {cv_scores_gb.mean():.4f} ± {cv_scores_gb.std():.4f}")
print()

✓ Gradient Boosting treinado
  Acurácia: 77.9%
  ROC AUC: 0.8413

  Cross-Validation (5-fold) ROC AUC: 0.8295 ± 0.0245



Resultados similares ao RF
__________________________

Avaliação

In [24]:
print(classification_report(y_test, y_pred_rf, target_names=['Não em Risco', 'Em Risco']))

print("\nMatriz de Confusão:")
cm = confusion_matrix(y_test, y_pred_rf)
print(cm)

# Calcular métricas adicionais
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred_rf)

print(f"\nMétricas Adicionais:")
print(f"  Sensibilidade (Recall): {sensitivity:.1%}")
print(f"  Especificidade: {specificity:.1%}")
print(f"  F1-Score: {f1:.4f}")
print()

              precision    recall  f1-score   support

Não em Risco       0.79      0.87      0.83        92
    Em Risco       0.77      0.66      0.71        62

    accuracy                           0.79       154
   macro avg       0.78      0.77      0.77       154
weighted avg       0.78      0.79      0.78       154


Matriz de Confusão:
[[80 12]
 [21 41]]

Métricas Adicionais:
  Sensibilidade (Recall): 66.1%
  Especificidade: 87.0%
  F1-Score: 0.7130



O modelo tem boa acurácia geral (~79%) e classifica melhor os alunos não em risco do que os em risco.
Especificidade alta (87%) → identifica bem quem não está em risco.
Sensibilidade moderada (66%) → ainda deixa escapar parte dos alunos em risco.
F1-score de 0.71 indica desempenho equilibrado, mas com espaço para melhorar na detecção de risco.
_________________________________

Modelo XGBoost - Modelo Campeão

In [29]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score

# Modelo XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

# Treinamento
xgb_model.fit(X_train_scaled, y_train)

# Predições
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Métricas principais
acc_xgb = accuracy_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print("✓ XGBoost treinado")
print(f"  Acurácia: {acc_xgb:.1%}")
print(f"  ROC AUC: {roc_auc_xgb:.4f}")
print()

# Cross-validation
cv_scores_xgb = cross_val_score(xgb_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"  Cross-Validation (5-fold) ROC AUC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")
print()

# Classification report
print(classification_report(y_test, y_pred_xgb, target_names=['Não em Risco', 'Em Risco']))

# Matriz de confusão
print("\nMatriz de Confusão:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

# Métricas adicionais
tn, fp, fn, tp = cm_xgb.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred_xgb)

print(f"\nMétricas Adicionais:")
print(f"  Sensibilidade (Recall): {sensitivity:.1%}")
print(f"  Especificidade: {specificity:.1%}")
print(f"  F1-Score: {f1:.4f}")

✓ XGBoost treinado
  Acurácia: 78.6%
  ROC AUC: 0.8592

  Cross-Validation (5-fold) ROC AUC: 0.8441 ± 0.0398

              precision    recall  f1-score   support

Não em Risco       0.81      0.84      0.82        92
    Em Risco       0.75      0.71      0.73        62

    accuracy                           0.79       154
   macro avg       0.78      0.77      0.78       154
weighted avg       0.78      0.79      0.78       154


Matriz de Confusão:
[[77 15]
 [18 44]]

Métricas Adicionais:
  Sensibilidade (Recall): 71.0%
  Especificidade: 83.7%
  F1-Score: 0.7273


Importancia das variaveis xgboost:

In [34]:
# Feature importance do XGBoost
feature_importance = pd.DataFrame({
    'Feature': features_list,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Features Mais Importantes:")
for idx, row in feature_importance.head(10).iterrows():
    print(f"  {row['Feature']:25s}: {row['Importance']:6.1%}")
print()

Top 10 Features Mais Importantes:
  IAN                      :  19.6%
  IPP                      :  18.1%
  INDICADORES_MEDIA        :   8.9%
  IPS                      :   8.6%
  IPV                      :   7.7%
  IEG_IDA_RATIO            :   6.6%
  INDE_IDA_RATIO           :   6.3%
  IAA                      :   6.3%
  DESALINHAMENTO_IAA_IDA   :   6.1%
  IDA                      :   6.0%



Importancia das variaveis RF

In [33]:
# Feature importance do Random Forest
feature_importance = pd.DataFrame({
    'Feature': features_list,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Features Mais Importantes:")
for idx, row in feature_importance.head(10).iterrows():
    print(f"  {row['Feature']:25s}: {row['Importance']:6.1%}")
print()

Top 10 Features Mais Importantes:
  IPP                      :  22.3%
  IPV                      :  11.1%
  INDICADORES_MEDIA        :  10.4%
  IPS                      :   9.0%
  INDE_IDA_RATIO           :   7.3%
  IDA                      :   7.3%
  IEG                      :   7.0%
  IAN                      :   6.9%
  IEG_IDA_RATIO            :   6.8%
  DESALINHAMENTO_IAA_IDA   :   6.6%



Salva Modelos:

In [35]:
# Salvar modelos
joblib.dump(rf_model, MODELS_DIR / "risk_model_rf.pkl")
joblib.dump(gb_model, MODELS_DIR / "risk_model_gb.pkl")
joblib.dump(xgb_model, MODELS_DIR / "risk_model_xgb.pkl")
joblib.dump(scaler, MODELS_DIR / "risk_model_scaler.pkl")

print(f"✓ Modelos salvos em {MODELS_DIR}")
print(f"  - risk_model_rf.pkl")
print(f"  - risk_model_gb.pkl")
print(f"  - risk_model_xgb.pkl")
print(f"  - risk_model_scaler.pkl")
print()

# Salvar metadados (XGBoost)
metadata = {
    'model_type': 'XGBoost',
    'features': features_list,
    'n_features': len(features_list),
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'accuracy': float(acc_xgb),
    'roc_auc': float(roc_auc_xgb),
    'sensitivity': float(sensitivity),
    'specificity': float(specificity),
    'f1_score': float(f1),
    'cv_mean': float(cv_scores_xgb.mean()),
    'cv_std': float(cv_scores_xgb.std()),
    'feature_importance': feature_importance.to_dict('list'),
    'training_years': [2022, 2023],
    'target_year': 2024,
}

with open(MODELS_DIR / "risk_model_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadados salvos: risk_model_metadata.json")
print()

✓ Modelos salvos em /content
  - risk_model_rf.pkl
  - risk_model_gb.pkl
  - risk_model_xgb.pkl
  - risk_model_scaler.pkl

✓ Metadados salvos: risk_model_metadata.json



Resumo

In [36]:
print(f"XGBoost:")
print(f"  • Acurácia: {acc_xgb:.1%}")
print(f"  • Sensibilidade: {sensitivity:.1%}")
print(f"  • Especificidade: {specificity:.1%}")
print(f"  • ROC AUC: {roc_auc_xgb:.4f}")
print(f"  • F1-Score: {f1:.4f}")
print(f"  • CV ROC AUC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")
print()

print(f"Top 3 Features:")
for idx, (_, row) in enumerate(feature_importance.head(3).iterrows(), 1):
    print(f"  {idx}. {row['Feature']}: {row['Importance']:.1%}")
print()

XGBoost:
  • Acurácia: 78.6%
  • Sensibilidade: 71.0%
  • Especificidade: 83.7%
  • ROC AUC: 0.8592
  • F1-Score: 0.7273
  • CV ROC AUC: 0.8441 ± 0.0398

Top 3 Features:
  1. IAN: 19.6%
  2. IPP: 18.1%
  3. INDICADORES_MEDIA: 8.9%

